# 7.2 Parameter Estimation

**Goal:** derive moment, likelihood, and posterior-mode estimators and understand information bounds.

**Prerequisites:** Modules 0–4 and earlier notebooks in this module; advanced notebooks also build on earlier modules 5–8. Read the section assumptions before using a method.

**Runtime:** Python 3.10+ with NumPy, SciPy, and Matplotlib. Install in your chosen environment with `python3 -m pip install numpy scipy matplotlib`, then open in JupyterLab or VS Code with a Python kernel. Use **Restart Kernel and Run All**. Each notebook is self-contained, includes saved outputs, uses synthetic data and a fixed local random seed, and needs no downloads. Numerical simulations illustrate results; they do not prove them or validate real-world assumptions.

## Module navigation

1. [Bayesian Statistics](1_Bayesian-Statistics.ipynb)
2. [Parameter Estimation](2_Parameter-Estimation.ipynb)
3. [Multivariate Statistics](3_Multivariate-Statistics.ipynb)
4. [Resampling Methods](4_Resampling-Methods.ipynb)
5. [Experimental Design](5_Experimental-Design.ipynb)

## Learning route

Read each definition and assumption, work through its calculation, run the code, then attempt the practice before viewing the solutions. Distinguish descriptive association, causal inference, and predictive evaluation throughout.

In [1]:
import math
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats, optimize, special, linalg
rng = np.random.default_rng(2026)
plt.rcParams.update({"figure.figsize": (7, 4), "axes.titlesize": 12, "font.size": 10})

## 1. Method of Moments

Match empirical moments with model moments and solve for parameters. For a Gamma(shape a,scale θ) model, mean aθ and variance aθ² suggest $\hat a=\bar x²/v_n$ and $\hat\theta=v_n/\bar x$ using empirical variance vₙ. Domain restrictions matter: zero variance or nonpositive means can make these formulas invalid for a nondegenerate gamma model.

Moment estimators are often easy to compute and useful as optimizer initial values, but may be inefficient or produce invalid parameter values in other families. The method does not validate that the chosen moments identify the entire distribution.

In [2]:
x=rng.gamma(shape=3,scale=2,size=300)
mean=x.mean();v=x.var(ddof=0)
print("Gamma moment estimates shape/scale:",mean**2/v,v/mean)

Gamma moment estimates shape/scale: 2.7529495225593292 2.17269380165888


## 2. Likelihood Function and Maximum Likelihood Estimation (MLE)

MLE maximizes $L(\theta)=p(D|\theta)$ over the valid parameter space; **log-likelihood** turns independent products into sums and improves numerical stability. Maximizing an unconstrained optimizer over invalid scales is not a valid likelihood analysis; reparameterize positive quantities with logs or impose bounds.

For IID Normal(μ,σ²), MLEs are sample mean and denominator-n variance. For IID Bernoulli, p-hat=s/n. Boundary MLEs can occur with all successes or failures. Consistency and asymptotic normality need identifiability and regularity, and are not universal for every likelihood or sampling process.

**Example:** optimize normal location and log-SD and compare with the analytic estimates.

In [3]:
def nll(params):
    mu,log_sd=params
    return -stats.norm.logpdf(x,loc=mu,scale=np.exp(log_sd)).sum()
fit=optimize.minimize(nll,[x.mean(),np.log(x.std())],method="BFGS")
print("Normal working-model MLE mean/SD:",fit.x[0],np.exp(fit.x[1]))
assert np.isclose(fit.x[0],x.mean(),atol=1e-4)
# These estimates fit a normal working model to gamma observations; fit alone does not imply correctness.

Normal working-model MLE mean/SD: 5.981316363944427 3.604936752524639


## 3. Maximum A Posteriori (MAP)

MAP maximizes log-likelihood plus log-prior. For Bernoulli data and Beta(a,b) prior with an interior posterior mode, $\hat p_{MAP}=(s+a-1)/(n+a+b-2)$. This formula requires both updated shape parameters >1; otherwise maxima can lie at boundaries or be nonunique. Posterior mean is $(s+a)/(n+a+b)$, a distinct estimator.

Gaussian coefficient priors connect MAP to L2 penalties and Laplace priors to L1 penalties under appropriate likelihood/scaling conventions. MAP reports a point, not posterior spread, and changes under nonlinear reparameterization because densities include Jacobians.

In [4]:
s,n,a,b=8,10,2,2
print("MLE:",s/n,"MAP:",(s+a-1)/(n+a+b-2),"posterior mean:",(s+a)/(n+a+b))

MLE: 0.8 MAP: 0.75 posterior mean: 0.7142857142857143


## 4. Fisher Information

For a regular model, the score is $U(\theta)=\partial\log p(D|\theta)/\partial\theta$. Fisher information can be $E[U²]$ or −E[second derivative] when regularity justifies equality. Independent information adds across observations. For n Bernoulli observations with 0<p<1, $I_n(p)=n/[p(1-p)]$.

Observed information uses the realized negative Hessian; expected information averages under the model. Multiple parameters use an information matrix. Boundary points, parameter-dependent supports, nonidentifiability, and misspecification can invalidate routine inverse-information approximations.

In [5]:
p=.3;n=100
information=n/(p*(1-p))
print("Bernoulli expected information:",information,"inverse:",1/information)

Bernoulli expected information: 476.1904761904762 inverse: 0.0021


## 5. Cramér–Rao Lower Bound

Under suitable regularity conditions, an unbiased scalar estimator of θ satisfies $Var(\hat\theta)\ge1/I_n(\theta)$. More generally an unbiased estimator of g(θ) has bound [g′(θ)]²/I. With nuisance parameters, use the appropriate matrix inverse element rather than blindly invert one diagonal entry.

**Worked example:** the Bernoulli sample proportion has variance p(1−p)/n, attaining this bound for interior p. The bound does not say every biased estimator has larger MSE: shrinkage can trade bias for lower variance. It is also not a guarantee that a finite-sample normal CI based on estimated information has exact coverage.

In [6]:
variance=p*(1-p)/n
assert np.isclose(variance,1/information)
print("Proportion variance matches regular unbiased information bound:",variance)

Proportion variance matches regular unbiased information bound: 0.0021


## AI/ML application and interpretation

MLE/MAP underpin model training and regularization; information-based uncertainty can be useful for interpretable probabilistic models. Check optimizer convergence, constraints, separation, conditioning, and model adequacy. A successful numerical fit is not evidence that the assumed likelihood reflects deployment data.

## Practice

1. What makes a log-likelihood easier to compute than a long likelihood product?
2. Why is a MAP formula derived for an interior beta mode invalid for every shape pair?
3. Does inverse Fisher information bound MSE for every biased estimator?
4. Does fitting a normal likelihood prove the observations are normal?

## Solutions

1. Sums reduce underflow and simplify derivatives without changing the maximizer.
2. Posterior densities may maximize at boundaries or have multiple boundary modes when shapes are ≤1.
3. No. The stated CRLB is a variance bound for an unbiased estimator under regularity; other bounds are needed for biased cases.
4. No. It supplies estimates within a working model whose adequacy requires separate examination.

**Readiness check:** identify the estimand, independent unit, assumptions, uncertainty source, and an important failure mode before using these methods.

**Next:** [Continue studying](3_Multivariate-Statistics.ipynb).